In [1]:
from __future__ import annotations
import os
import json
import importlib
import argparse
from functools import partial

import random
import numpy as np
import pickle

import torch

from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

from core import mcts_search_v51_tree
from core import mcts_search_v21_tree
from core import mcts_search_v71_tree
from core import mcts_search_v81_tree
from core import mcts_search_mini_v51
from core import mcts_search_mini_v12
from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k

# from core.llm_engine import rm_engine
# from core.llms import rm_generate
# import logging
# logging.basicConfig(format='%(message)s', level=logging.ERROR)

INFO 07-25 07:41:39 [__init__.py:244] Automatically detected platform cuda.


In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
llm_total_gpu = 0.4
llm_gpu_memory_utilization = 0.2
# llm_vllm = LLM(
#     model = llm_dir,
#     tensor_parallel_size=1,
#     gpu_memory_utilization = 0.7,  # Utilize 50% of GPU memory
#     # enable_prefix_caching=True,  # V100 doesn't support enable_prefix_caching 
#     # enable_chunked_prefill=False, # and enable_chunked_prefill
#     max_model_len = 5000,
#     dtype = "float16",
#     seed = config.seed)

llm_vllm = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)

print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))



INFO 07-25 07:41:56 [config.py:823] This model supports multiple tasks: {'generate', 'embed', 'reward', 'score', 'classify'}. Defaulting to 'generate'.
WARNING 07-25 07:41:56 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 07-25 07:41:56 [arg_utils.py:1642] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 07-25 07:41:56 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 07-25 07:41:56 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_par

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-25 07:42:00 [default_loader.py:272] Loading weights took 1.36 seconds
INFO 07-25 07:42:01 [model_runner.py:1203] Model loading took 2.3185 GiB and 1.583081 seconds
INFO 07-25 07:42:01 [worker.py:294] Memory profiling takes 0.51 seconds
INFO 07-25 07:42:01 [worker.py:294] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.20) = 6.35GiB
INFO 07-25 07:42:01 [worker.py:294] model weights take 2.32GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.19GiB; the rest of the memory reserved for KV Cache is 2.75GiB.
INFO 07-25 07:42:02 [executor_base.py:113] # cuda blocks: 5631, # CPU blocks: 32768
INFO 07-25 07:42:02 [executor_base.py:118] Maximum concurrency for 5000 tokens per request: 18.02x
INFO 07-25 07:42:08 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 7.07 seconds
#--- memory: 5.07647705078125


In [5]:
llm_vllm_embeds = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    task="embed",
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_total_gpu-llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

WARNING 07-25 07:42:08 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 07-25 07:42:08 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 07-25 07:42:08 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_pr

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-25 07:42:10 [default_loader.py:272] Loading weights took 1.35 seconds
INFO 07-25 07:42:11 [model_runner.py:1203] Model loading took 2.3029 GiB and 1.388627 seconds
#--- memory: 7.379390716552734


In [6]:
prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

#--- memory: 22.336918354034424


In [7]:
stop

NameError: name 'stop' is not defined

In [8]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)

1: 43
2: 90
3: 105
4: 128
5: 134


In [9]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 3            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0

# mcts parameter
config.num_batches = 1
config.batch_budget = config.num_batches*config.max_depths 
config.num_phases = config.batch_budget

config.lam = 10 
config.normalize_embeds = True

config.cpuct_root = 0
config.cpuct_leaf = 0
config.cpuct = 10000
config.ds_beta = 1.0
config.ds_alpha = 10.0
config.use_ppl = True

config.version = "v51"



In [10]:
level = 4                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
num_questions = 1
num_trials = 1
print(f"num_questions = {num_questions}")
print(f"num_trials = {num_trials}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
q_idx = 53
batch_of_questions = [data_by_levels[level][q_idx]['problem']]
batch_of_answers = [data_by_levels[level][q_idx]['answer']]
print(batch_of_questions)
print(batch_of_answers)

num_questions = 1
num_trials = 1
['Find $\\sin 20^\\circ \\sin 40^\\circ \\sin 60^\\circ \\sin 80^\\circ \\sin 100^\\circ \\sin 120^\\circ \\sin 140^\\circ \\sin 160^\\circ.$']
['\\frac{9}{256}']


In [11]:
config.n = 4
config.max_depths = 7
config.num_batches = 3
config.num_phases = config.n**(config.max_depths-1)
# config.batch_budget = config.num_batches**config.max_depths
# config.batch_budget = config.n**config.max_depths -1
# config.max
print(config.num_phases)
print(config.max_depths)
print(config.batch_budget)

import logging
logging.basicConfig(format='%(message)s', level=logging.CRITICAL+1)

4096
7
3


In [12]:
importlib.reload(mcts_search_v51_tree)
importlib.reload(mcts_search_v21_tree)
importlib.reload(mcts_search_v71_tree)
importlib.reload(mcts_search_v81_tree)
# importlib.reload(mcts_search_v51_mini)

<module 'core.mcts_search_v81_tree' from '/home/u20/tnguyen9210/tnn1/LLMs/llm-reasoning-methods/core/mcts_search_v81_tree.py'>

In [13]:

trial_idx = 0
config_name = f"mcts--n-{config.n}--d-{config.max_depths}--normalize-{config.normalize_embeds}--level-{level}--qidx-{q_idx}"
torch.manual_seed(100000+trial_idx)
torch.cuda.manual_seed(100000+trial_idx)

for question in batch_of_questions:
    agent = mcts_search_v81_tree.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v81_tree.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)
        

In [ ]:
print(tree_dict['0.0.0.1']['text'])
print(tree_dict['0.0.1']['text'])
print(tree_dict['0.0.2']['text'])
print(tree_dict['0.0.3']['text'])

In [ ]:
stop

In [14]:
with open(f"results/tree_{config_name}--trial-{trial_idx}.pkl", 'wb') as fout:
    pickle.dump(tree_dict, fout)

In [ ]:
trial_idx = 0
config_name = f"mcts--n-{config.n}--d-{config.max_depths}--normalize-{config.normalize_embeds}--level-{level}"
with open(f"results/tree_{config_name}--trial-{trial_idx}.pkl", 'rb') as fout:
    tree_dict = pickle.load(fout)

In [ ]:
config.n = 4
config.max_depths = 7
# config.num_phases = config.n**(config.max_depths-1)
config.num_phases = 10
config.num_batches = 5
config.batch_budget = config.num_batches*config.max_depths
print(config.batch_budget)
importlib.reload(mcts_search_mini_v51)
importlib.reload(mcts_search_mini_v12)

In [ ]:

for question in batch_of_questions:
    agent = mcts_search_mini_v51.MCTS(config=config, question=question)
    tree_dict_v51, tag_list_v51 = mcts_search_mini_v51.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

In [ ]:
print(tree_dict['0.0']['text'])
print(tree_dict['0.1']['text'])
print(tree_dict['0.2']['text'])
print(tree_dict['0.3']['text'])

In [ ]:

for question in batch_of_questions:
    agent = mcts_search_mini_v12.MCTS(config=config, question=question)
    tree_dict_v12, tag_list_v12 = mcts_search_mini_v12.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

In [ ]:
print(tree_dict['0.0']['text'])
print(tree_dict['0.1']['text'])
print(tree_dict['0.2']['text'])
print(tree_dict['0.3']['text'])
# print(tree_dict['0.0.0.0.0.0.0.0.1'])
# print(tree_dict['0.0.0.0.0.0.0.0.2'])
print()
# print(tree_dict['0.1.3.2.1.0.1'])
# print(tree_dict['0.1.3.2.1.0.3'])

In [ ]:
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for question in batch_of_questions:
    agent = mcts_search_v51_tree.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v51_tree.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)


In [ ]:
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for question in batch_of_questions:
    agent = mcts_search_v71_tree.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v71_tree.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)

In [ ]:
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for question in batch_of_questions:
    agent = mcts_search_v81.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v81.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)

In [ ]:
torch.manual_seed(100000+0)
torch.cuda.manual_seed(100000+0)

for question in batch_of_questions:
    agent = mcts_search_v21_tree.MCTS(config=config, question=question)
    tree_dict, tag_list = mcts_search_v21_tree.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)
    # print(agent_completions)


In [ ]:
# for idx, node in enumerate(agent_completions):
#     print(f"\n-> idx = {idx}")
#     print(node.state["text"])
# print(agent_completions)
# for (key, value) in tree_dict.items():
#     print(f"\n->")
#     print(key)
#     print(value)
print(tree_dict['0.1.3.2.1.0.0'])
print()
print(tree_dict['0.1.3.2.1.0.1'])
print(tree_dict['0.1.3.2.1.0.3'])